# 📐 Chapter 3：矩陣與線性變換 (Matrices and Linear Transformations)

> 對應 3Blue1Brown —《線性代數的本質》第三章

**核心觀念：** 矩陣不是一堆數字，它是一個**變換**。每個 2×2 矩陣都在告訴你：「我把空間變成了什麼樣子。」

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# ── Helper: draw grid lines ──────────────────────────────────────────
def draw_grid(ax, matrix=None, n=8, color='lightblue', alpha=0.4, lw=0.8):
    """
    Draw a grid of lines on `ax`.
    If `matrix` is given (2x2), transform every grid point through it.
    Grid spans from -n to n.
    """
    t = np.linspace(-n, n, 300)
    for k in range(-n, n + 1):
        # Horizontal line: y = k
        xs_h, ys_h = t, np.full_like(t, k)
        # Vertical line: x = k
        xs_v, ys_v = np.full_like(t, k), t
        if matrix is not None:
            M = np.array(matrix, dtype=float)
            coords_h = M @ np.vstack([xs_h, ys_h])
            coords_v = M @ np.vstack([xs_v, ys_v])
            xs_h, ys_h = coords_h[0], coords_h[1]
            xs_v, ys_v = coords_v[0], coords_v[1]
        ax.plot(xs_h, ys_h, color=color, alpha=alpha, lw=lw)
        ax.plot(xs_v, ys_v, color=color, alpha=alpha, lw=lw)

# ── Helper: draw basis vectors ───────────────────────────────────────
def draw_basis(ax, matrix=None, scale=1.0):
    """
    Draw î and ĵ (or their transformed versions) as arrows from the origin.
    """
    i_hat = np.array([1, 0], dtype=float)
    j_hat = np.array([0, 1], dtype=float)
    if matrix is not None:
        M = np.array(matrix, dtype=float)
        i_hat = M @ i_hat
        j_hat = M @ j_hat
    ax.annotate('', xy=i_hat, xytext=[0, 0],
                arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2.5))
    ax.annotate('', xy=j_hat, xytext=[0, 0],
                arrowprops=dict(arrowstyle='->', color='#2ecc71', lw=2.5))
    offset = 0.15 * scale
    ax.text(i_hat[0] + offset, i_hat[1] + offset, r'$\hat{\imath}$',
            color='#e74c3c', fontsize=13, fontweight='bold')
    ax.text(j_hat[0] + offset, j_hat[1] + offset, r'$\hat{\jmath}$',
            color='#2ecc71', fontsize=13, fontweight='bold')

# ── Helper: draw a vector arrow ──────────────────────────────────────
def draw_vector(ax, v, origin=(0, 0), color='blue', label=None, lw=2.0):
    """Draw a single vector arrow from `origin` to `origin + v`."""
    v = np.asarray(v, dtype=float)
    o = np.asarray(origin, dtype=float)
    ax.annotate('', xy=o + v, xytext=o,
                arrowprops=dict(arrowstyle='->', color=color, lw=lw))
    if label:
        mid = o + v
        ax.text(mid[0] + 0.1, mid[1] + 0.1, label,
                color=color, fontsize=11, fontweight='bold')

# ── Helper: draw the unit square ─────────────────────────────────────
def draw_unit_square(ax, matrix=None, color='#3498db', alpha=0.15):
    """Draw the unit square [0,1]x[0,1], optionally transformed."""
    corners = np.array([[0, 0], [1, 0], [1, 1], [0, 1]], dtype=float)
    if matrix is not None:
        M = np.array(matrix, dtype=float)
        corners = (M @ corners.T).T
    poly = plt.Polygon(corners, closed=True, facecolor=color, alpha=alpha,
                        edgecolor=color, lw=1.5)
    ax.add_patch(poly)

# ── Helper: standard axis formatting ─────────────────────────────────
def fmt_ax(ax, lim=4, title=''):
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.axhline(0, color='gray', lw=0.5)
    ax.axvline(0, color='gray', lw=0.5)
    ax.set_title(title, fontsize=12)
    ax.grid(False)

print("Helper functions loaded ✓")

## Part 1：什麼是線性變換？

**線性變換** = 一個滿足以下兩個條件的函數 $L$：

1. **可加性 (Additivity)：** $L(\vec{v} + \vec{w}) = L(\vec{v}) + L(\vec{w})$
2. **齊次性 (Homogeneity)：** $L(c\vec{v}) = c \cdot L(\vec{v})$

### 直覺理解
- 變換後，所有**格線保持平行且等距**
- **原點不動**（始終映射到自己）

簡單來說：直線變換後還是直線，平行線還是平行線，原點釘在原地。

In [ ]:
# Part 1: Before vs after a linear transformation
M = np.array([[1, -0.5],
              [0.5, 1]])  # A simple transformation matrix

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# -- Before --
ax = axes[0]
draw_grid(ax, matrix=None, color='cornflowerblue', alpha=0.5)
draw_basis(ax)
draw_unit_square(ax)
fmt_ax(ax, title='變換前 (Before)')

# -- After --
ax = axes[1]
draw_grid(ax, matrix=M, color='cornflowerblue', alpha=0.5)
draw_basis(ax, matrix=M)
draw_unit_square(ax, matrix=M)
fmt_ax(ax, title='變換後 (After)')

fig.suptitle('線性變換：格線仍然平行且等距，原點不動', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Part 2：矩陣 = 「基底去了哪裡」

### 核心洞見

一個線性變換**完全由 $\hat{\imath}$ 和 $\hat{\jmath}$ 變換後的位置決定**。

$$
M = \begin{bmatrix} | & | \\ L(\hat{\imath}) & L(\hat{\jmath}) \\ | & | \end{bmatrix}
$$

- 矩陣的**第一行 (column)** = $\hat{\imath}$ 的新位置
- 矩陣的**第二行 (column)** = $\hat{\jmath}$ 的新位置

> 只要你知道基底去了哪裡，整個變換就確定了！

In [ ]:
# Part 2: Four classic transformations — where do the basis vectors go?

theta = np.pi / 6  # 30 degrees

transforms = {
    '旋轉 30° (Rotation)': np.array([[np.cos(theta), -np.sin(theta)],
                                       [np.sin(theta),  np.cos(theta)]]),
    '剪切 (Shear)':         np.array([[1, 1],
                                       [0, 1]]),
    '拉伸 (Scale)':         np.array([[2, 0],
                                       [0, 0.5]]),
    '反射 (Reflect y=x)':   np.array([[0, 1],
                                       [1, 0]]),
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

for ax, (name, M) in zip(axes, transforms.items()):
    draw_grid(ax, matrix=M, color='cornflowerblue', alpha=0.4)
    draw_basis(ax, matrix=M)
    draw_unit_square(ax, matrix=M)
    fmt_ax(ax, title=name)
    # Show matrix values in the corner
    mat_str = (f"[{M[0,0]:+.2f}  {M[0,1]:+.2f}]\n"
               f"[{M[1,0]:+.2f}  {M[1,1]:+.2f}]")
    ax.text(-3.8, 3.3, mat_str, fontsize=9, fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

fig.suptitle('四種經典線性變換：觀察 î 和 ĵ 去了哪裡', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Part 3：矩陣 × 向量 = 線性變換

任何向量都可以寫成基底的線性組合：

$$
\vec{v} = x\hat{\imath} + y\hat{\jmath}
$$

套用線性變換後：

$$
L(\vec{v}) = x \cdot L(\hat{\imath}) + y \cdot L(\hat{\jmath})
$$

所以 $M \vec{v}$ 的計算方式就是：**用「新基底的座標」重新組合**。

$$
\begin{bmatrix} a & b \\ c & d \end{bmatrix}
\begin{bmatrix} x \\ y \end{bmatrix}
= x \begin{bmatrix} a \\ c \end{bmatrix}
+ y \begin{bmatrix} b \\ d \end{bmatrix}
$$

In [ ]:
# Part 3: Decompose M @ v into x * new_i + y * new_j

M = np.array([[2, 1],
              [1, 1]])
v = np.array([2, 1])

new_i = M[:, 0]   # first column
new_j = M[:, 1]   # second column
x, y = v[0], v[1]

result = M @ v  # should equal x * new_i + y * new_j

fig, ax = plt.subplots(figsize=(8, 8))
draw_grid(ax, matrix=M, color='cornflowerblue', alpha=0.3)
fmt_ax(ax, lim=6, title=r'分解過程：$M\vec{v} = x \cdot L(\hat{\imath}) + y \cdot L(\hat{\jmath})$')

# Step 1: draw new basis vectors
draw_vector(ax, new_i, color='#e74c3c', label=r'$L(\hat{\imath})$', lw=1.5)
draw_vector(ax, new_j, color='#2ecc71', label=r'$L(\hat{\jmath})$', lw=1.5)

# Step 2: draw scaled components
draw_vector(ax, x * new_i, color='#e74c3c', label=f'{x}×L(î)=({x*new_i[0]},{x*new_i[1]})',
            lw=2.5)
draw_vector(ax, y * new_j, origin=x * new_i, color='#2ecc71',
            label=f'{y}×L(ĵ)=({y*new_j[0]},{y*new_j[1]})', lw=2.5)

# Step 3: draw the final result
draw_vector(ax, result, color='#9b59b6', label=f'Mv = ({result[0]},{result[1]})', lw=3)

# Annotation box
info = (f"v = ({v[0]}, {v[1]})\n"
        f"M = [[{M[0,0]}, {M[0,1]}], [{M[1,0]}, {M[1,1]}]]\n"
        f"Mv = {x}×[{new_i[0]},{new_i[1]}] + {y}×[{new_j[0]},{new_j[1]}]\n"
        f"   = [{x*new_i[0]},{x*new_i[1]}] + [{y*new_j[0]},{y*new_j[1]}]\n"
        f"   = [{result[0]},{result[1]}]")
ax.text(-5.8, -2.5, info, fontsize=10, fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

plt.tight_layout()
plt.show()

## Part 4：經典變換動物園

一口氣看完常見的線性變換！觀察格線、基底向量和單位正方形的變化。

In [ ]:
# Part 4a: Rotation zoo — 30°, 90°, 180°

def rotation_matrix(deg):
    """Return 2x2 rotation matrix for given degrees."""
    rad = np.radians(deg)
    return np.array([[np.cos(rad), -np.sin(rad)],
                     [np.sin(rad),  np.cos(rad)]])

angles = [30, 90, 180]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

for ax, deg in zip(axes, angles):
    M = rotation_matrix(deg)
    draw_grid(ax, matrix=M, color='cornflowerblue', alpha=0.4)
    draw_basis(ax, matrix=M)
    draw_unit_square(ax, matrix=M)
    fmt_ax(ax, title=f'旋轉 {deg}°')
    mat_str = f"[{M[0,0]:+.2f}  {M[0,1]:+.2f}]\n[{M[1,0]:+.2f}  {M[1,1]:+.2f}]"
    ax.text(-3.8, 3.3, mat_str, fontsize=9, fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

fig.suptitle('旋轉 (Rotation)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Part 4b: Shear zoo — shear-x vs shear-y

shears = {
    '水平剪切 (Shear X)': np.array([[1, 1],
                                      [0, 1]]),
    '垂直剪切 (Shear Y)': np.array([[1, 0],
                                      [1, 1]]),
}

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

for ax, (name, M) in zip(axes, shears.items()):
    draw_grid(ax, matrix=M, color='cornflowerblue', alpha=0.4)
    draw_basis(ax, matrix=M)
    draw_unit_square(ax, matrix=M)
    fmt_ax(ax, title=name)
    mat_str = f"[{M[0,0]:+.0f}  {M[0,1]:+.0f}]\n[{M[1,0]:+.0f}  {M[1,1]:+.0f}]"
    ax.text(-3.8, 3.3, mat_str, fontsize=10, fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

fig.suptitle('剪切 (Shear)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Part 4c: Reflection zoo — reflect across x-axis vs y-axis

reflections = {
    '對 x 軸反射 (Reflect x)': np.array([[ 1,  0],
                                            [ 0, -1]]),
    '對 y 軸反射 (Reflect y)': np.array([[-1,  0],
                                            [ 0,  1]]),
}

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

for ax, (name, M) in zip(axes, reflections.items()):
    draw_grid(ax, matrix=M, color='cornflowerblue', alpha=0.4)
    draw_basis(ax, matrix=M)
    draw_unit_square(ax, matrix=M)
    fmt_ax(ax, title=name)
    mat_str = f"[{M[0,0]:+.0f}  {M[0,1]:+.0f}]\n[{M[1,0]:+.0f}  {M[1,1]:+.0f}]"
    ax.text(-3.8, 3.3, mat_str, fontsize=10, fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

fig.suptitle('反射 (Reflection)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Part 5：練習區

試試看！修改下方的矩陣數值，觀察不同的線性變換效果。

一些有趣的矩陣可以嘗試：
- `[[2, 0], [0, 2]]` — 均勻放大 2 倍
- `[[1, 0], [0, 0]]` — 投影到 x 軸（壓扁到一條線！）
- `[[0, -1], [1, 0]]` — 旋轉 90°
- `[[1, 2], [3, 4]]` — 隨意組合
- `[[0, 0], [0, 0]]` — 全部壓到原點（行列式 = 0）

In [ ]:
# ✏️ Part 5: Try your own matrix! Edit the values below.

# === EDIT HERE ===
my_matrix = np.array([[1, 2],
                      [0, 1]])
# =================

det = np.linalg.det(my_matrix)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Before
ax = axes[0]
draw_grid(ax, matrix=None, color='cornflowerblue', alpha=0.5)
draw_basis(ax)
draw_unit_square(ax)
fmt_ax(ax, title='變換前 (Original)')

# After
ax = axes[1]
draw_grid(ax, matrix=my_matrix, color='cornflowerblue', alpha=0.5)
draw_basis(ax, matrix=my_matrix)
draw_unit_square(ax, matrix=my_matrix)
fmt_ax(ax, lim=5, title='變換後 (Transformed)')

mat_str = (f"M = [[{my_matrix[0,0]:.2f}, {my_matrix[0,1]:.2f}],\n"
           f"     [{my_matrix[1,0]:.2f}, {my_matrix[1,1]:.2f}]]\n"
           f"det(M) = {det:.2f}")
ax.text(-4.8, -3.5, mat_str, fontsize=10, fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

fig.suptitle('你的自定義變換', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()